# Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver.erp_product_category")

# Read from bronze

In [0]:
try:
    df = spark.table("workspace.bronze.erp_px_cat_g1v2_raw")
except Exception as e:
    logger.error(f"Failed to read Bronze table: {e}")
    raise

# Data transformations

## Renaming columns

In [0]:
RENAME_MAP = {
    "ID": "category_id",
    "CAT": "category",
    "SUBCAT": "subcategory",
    "MAINTENANCE": "maintenance_flag"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

## Normalize Maintenance Flag to Boolean

In [0]:
df = df.withColumn(
    "maintenance_flag",
    F.when(F.upper(F.col("maintenance_flag")) == "YES", F.lit(True))
    .when(F.upper(F.col("maintenance_flag")) == "NO", F.lit(False))
    .otherwise(None)
)

# Write into Silver Table

In [0]:
try:
    df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.erp_product_category")
except Exception as e:
    logger.error(f"Failed to write Silver table: {e}")
    raise